In [1]:
# Librairies
import pickle
import healpy as hp
import numpy as np
import plotly.graph_objects as go

# Modules 
import netcdf_read_write as nrw
import training_data_generation as tdg

In [2]:
with open("results/simu_3.pkl", "rb") as f:
    all_tree = pickle.load(f)

num_tree = len(all_tree)

In [ ]:
def read_data(parameters, dataset_params):
    def imap_norm(intensity_map):
        avg_flux = np.mean(intensity_map)
        return intensity_map / avg_flux - 1.0, avg_flux

    def imap2modes(intensity_map_normalized, lmax):
        modes_complex = hp.sphtfunc.map2alm(intensity_map_normalized, lmax=lmax)
        return modes_complex.real, modes_complex.imag

    def alms2rms(real_modes, imag_modes, lmax):
        n_m0 = lmax + 1  # correction : lmax+1 termes m=0
        pwr_spec_m0   = np.sum(real_modes[:n_m0]**2 + imag_modes[:n_m0]**2)
        pwr_spec_rest = np.sum((real_modes[n_m0:]**2 + imag_modes[n_m0:]**2) * 2)
        return np.sqrt((pwr_spec_m0 + pwr_spec_rest) / (4.0 * np.pi))
    
    dataset = {}
    intensity_map = parameters["intensity"] * (dataset_params['illumination_evaluation_radii'] / 10000.0)**2
    dataset["nside"] = hp.get_nside(intensity_map) # 64 dans le fichier input d'ifriit
    print(dataset["nside"])
    intensity_map_normalized, dataset["avg_flux"] = imap_norm(intensity_map)
    dataset["real_modes"], dataset["imag_modes"] = imap2modes(intensity_map_normalized, dataset_params["LMAX"])
    dataset["rms"] = alms2rms(dataset["real_modes"], dataset["imag_modes"], dataset_params["LMAX"])
    dataset["power_deposited"] = dataset["avg_flux"] * 4.0 * np.pi
    return dataset

main_dir = "../Data/Data_run5"
sys_params = tdg.define_system_params(main_dir)
dataset_params = nrw.read_general_netcdf(main_dir + "/" + sys_params["dataset_params_filename"])
num_perturbations = dataset_params["num_perturbations"]


for i in range(num_tree):
    p_in_z1z2_beam_all = all_tree[i]
    for j in range(num_perturbations):
        all_tree[i][j]['dataset'] = read_data(p_in_z1z2_beam_all[j]['p_in_z1z2_beam_all'], dataset_params)

In [4]:
depth_values = []

seuil = 0.01

# Parcours de la structure imbriquée all_tree
for i in range(num_tree):
    for j in range(num_perturbations):
        dataset = all_tree[i][j]['dataset']
        if dataset['rms']> seuil or j==num_perturbations-1:
            depth_values.append(j)
            break

depths, occurrences = np.unique(depth_values, return_counts=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=depths,
    y=occurrences,
    marker=dict(
        color=occurrences,          # Colore la barre selon le nombre d'occurrences
        colorscale='Plasma',        # Palette de couleur distincte
        line=dict(color='black', width=1)
    ),
    text=occurrences,               # Affiche le nombre directement sur/au-dessus de la barre
    textposition='auto',
    name='Occurrences'
))

# Configuration du layout
fig.update_layout(
    title=f"Distribution des profondeurs (Depth) pour RMS > {seuil}",
    xaxis=dict(
        title="Depth (Profondeur / Indice de perturbation)",
        tickmode='linear',          # Force l'affichage de chaque entier sur l'axe X
        dtick=1
    ),
    yaxis=dict(
        title="Nombre d'occurrences (Fréquence)"
    ),
    width=800,
    height=500,
    template="plotly_white"
)

fig.show()